# Radar-coordinate burst interferogram (ISCE2 topsApp-style, ISCE3 directly)

An aside notebook showing how to form a Sentinel-1 interferogram in **radar coordinates** instead of the geocoded path that the rest of `sweets` (and COMPASS) uses. This mirrors what ISCE2's `topsApp.py` does on a stripped-down scale: pick a few bursts in one subswath, co-register a secondary acquisition to a reference acquisition by way of `rdr2geo` -> `geo2rdr` -> `resamp_slc`, multiply them to form per-burst wrapped interferograms, then stitch the bursts together along azimuth.

Why ISCE3 and not COMPASS here? COMPASS does expose a `s1_cslc.py --grid radar` workflow built on these same ISCE3 primitives, but most of the value COMPASS adds (timing/atmospheric correction LUTs, layover/shadow masks, geometry products, HDF5 metadata wrap-up) is wired into the geocoded path. For a radar-coord interferogram the ISCE3 calls themselves are the interesting part, so this notebook drives them directly. If you want to plug COMPASS in, swap each block below for the corresponding `s1_rdr2geo.run` / `s1_geo2rdr.run` / `s1_resample.run` invocation -- the underlying ISCE3 call sequence is identical.

**Scope**: 4 contiguous IW2 bursts on Sentinel-1 descending track 71 over Los Angeles, two repeat passes ~12 days apart. End product is one wrapped interferogram in radar coordinates, ~1500 azimuth lines x 4 burst-lengths.

**Not for production**: there is no ionospheric correction, no precise topo-phase removal beyond what `flatten=True` does inside ISCE3's resamp, no ESD (enhanced spectral diversity) refinement of the azimuth offsets, and no proper handling of the deramp/reramp during burst overlap. The geocoded `sweets` workflow uses dolphin and remains the right tool for actual time-series work.

## Earthdata credentials

Same as the other example notebooks: a `~/.netrc` entry for `urs.earthdata.nasa.gov` is required so that `burst2safe`, `eof`, and `sardem` can pull bursts, precise orbits, and the Copernicus DEM.

## Inputs

Pick the AOI, the two acquisition dates, the relative orbit / track, and the subswath. The AOI is intentionally tall in latitude so that `burst2safe` returns 4 contiguous bursts in IW2.

In [ ]:
from pathlib import Path
from datetime import datetime

WORK_DIR = Path("./example_radar_coord_ifg").resolve()
WORK_DIR.mkdir(exist_ok=True)

# AOI sized to grab ~4 IW2 bursts on descending track 71 over LA.
AOI = (-118.55, 33.55, -118.30, 34.00)  # west, south, east, north
TRACK = 71
SWATH = 2  # IW2
POL = "VV"

# Two repeat passes ~12 days apart.
REF_DATE = datetime(2025, 12, 6)
SEC_DATE = datetime(2025, 12, 18)

SAFE_DIR = WORK_DIR / "safes"
ORBIT_DIR = WORK_DIR / "orbits"
DEM_PATH = WORK_DIR / "dem.tif"
PROC_DIR = WORK_DIR / "processing"
for d in (SAFE_DIR, ORBIT_DIR, PROC_DIR):
    d.mkdir(parents=True, exist_ok=True)
print(WORK_DIR)

## Download SAFEs, orbits, and DEM

Reuses `sweets.download.BurstSearch` (which wraps `burst2safe`) and `sweets._orbit.download_orbits` (which wraps `eof`). Each call is idempotent: re-running the cell skips work already on disk.

In [ ]:
from sweets.download import BurstSearch
from sweets._orbit import download_orbits
from sweets.dem import create_dem

search = BurstSearch(
    bbox=AOI,
    start=REF_DATE,
    end=SEC_DATE,
    track=TRACK,
    swaths=[f"IW{SWATH}"],
    polarizations=[POL],
    flight_direction="DESCENDING",
    out_dir=SAFE_DIR,
)
safes = search.existing_safes() or search.download()
assert len(safes) >= 2, f"need at least 2 SAFEs, got {len(safes)}: {safes}"

# Pair SAFEs to dates by the start-time field in the filename.
def safe_date(safe: Path) -> datetime:
    return datetime.strptime(safe.name.split("_")[5][:8], "%Y%m%d")

ref_safe = next(s for s in safes if safe_date(s) == REF_DATE)
sec_safe = next(s for s in safes if safe_date(s) == SEC_DATE)
print(f"reference: {ref_safe.name}")
print(f"secondary: {sec_safe.name}")

orbits = download_orbits(SAFE_DIR, ORBIT_DIR)
ref_orbit = next(o for o in orbits if REF_DATE.strftime("%Y%m%d") in o.name)
sec_orbit = next(o for o in orbits if SEC_DATE.strftime("%Y%m%d") in o.name)

# Buffer DEM ~0.5 deg around AOI so rdr2geo never asks for a pixel off the edge.
dem_bbox = (AOI[0] - 0.5, AOI[1] - 0.5, AOI[2] + 0.5, AOI[3] + 0.5)
create_dem(DEM_PATH, dem_bbox)
print(f"DEM: {DEM_PATH}")

## Load bursts via `s1reader`

`s1reader.load_bursts(safe, orbit, swath, pol)` returns one `Sentinel1BurstSlc` per burst in the subswath. Each carries its own ISCE3 radar grid, orbit, doppler, sensing time, and a `slc_to_file` / `slc_to_vrt_file` helper for materializing the complex SLC samples on disk.

Pair reference and secondary by `burst_id` (an `(esa_track, relative_burst_id, subswath)` tuple) and keep all 4 of them sorted in azimuth time.

In [ ]:
import s1reader

ref_bursts = s1reader.load_bursts(str(ref_safe), str(ref_orbit), SWATH, POL)
sec_bursts = s1reader.load_bursts(str(sec_safe), str(sec_orbit), SWATH, POL)

ref_by_id = {str(b.burst_id): b for b in ref_bursts}
sec_by_id = {str(b.burst_id): b for b in sec_bursts}
common_ids = sorted(set(ref_by_id) & set(sec_by_id))
print(f"{len(common_ids)} common bursts: {common_ids}")

# Take the 4 in azimuth-time order. If the AOI yielded more, trim to the
# central 4 so the output stays small.
common_ids.sort(key=lambda bid: ref_by_id[bid].sensing_start)
if len(common_ids) > 4:
    start = (len(common_ids) - 4) // 2
    common_ids = common_ids[start : start + 4]
burst_pairs = [(ref_by_id[bid], sec_by_id[bid]) for bid in common_ids]
print(f"using {len(burst_pairs)} bursts: {[bid for bid in common_ids]}")

## Per-burst co-registration: `rdr2geo` -> `geo2rdr` -> `resamp_slc`

For each burst pair the reference SLC stays on its native radar grid and we resample the secondary onto that same grid. The three ISCE3 calls are:

1. **`Rdr2Geo.topo`** on the reference burst writes per-pixel `(x, y, z)` (lon, lat, height) rasters by intersecting each radar pixel's line-of-sight with the DEM. These are the topo layers.
2. **`Geo2Rdr.geo2rdr`** takes those topo layers and asks: for each reference radar pixel, where does that ground point land in the secondary's radar grid? It writes `range.off` and `azimuth.off`.
3. **`ResampSlc.resamp`** uses those offsets to interpolate the secondary SLC onto the reference's radar grid, with `flatten=True` to subtract the reference-vs-secondary range delay phase ramp.

After step 3 the resampled secondary is geometrically aligned with the reference at sub-pixel precision in radar coordinates, and `ref * conj(coreg_sec)` is the wrapped interferogram for that burst.

In [ ]:
import isce3
from osgeo import gdal

DEM_RASTER = isce3.io.Raster(str(DEM_PATH))
EPSG = DEM_RASTER.get_epsg()
ELLIPSOID = isce3.core.make_projection(EPSG).ellipsoid


def run_rdr2geo(ref_burst, out_dir: Path) -> Path:
    """Write reference SLC + (x, y, z) topo layers and return topo.vrt path."""
    out_dir.mkdir(parents=True, exist_ok=True)
    ref_slc = out_dir / "reference.slc.tif"
    if not ref_slc.exists():
        ref_burst.slc_to_file(str(ref_slc), "GTiff")

    topo_vrt = out_dir / "topo.vrt"
    if topo_vrt.exists():
        return topo_vrt

    rdr_grid = ref_burst.as_isce3_radargrid()
    rdr2geo = isce3.geometry.Rdr2Geo(
        rdr_grid,
        ref_burst.orbit,
        ELLIPSOID,
        isce3.core.LUT2d(),
        threshold=1e-8,
        numiter=25,
        extraiter=10,
        lines_per_block=1000,
    )
    layers = {}
    for name in ("x", "y", "z"):
        layers[name] = isce3.io.Raster(
            str(out_dir / f"{name}.tif"),
            rdr_grid.width,
            rdr_grid.length,
            1,
            gdal.GDT_Float64,
            "GTiff",
        )
    rdr2geo.topo(
        DEM_RASTER,
        x_raster=layers["x"],
        y_raster=layers["y"],
        height_raster=layers["z"],
    )
    # Flush before stacking into a VRT.
    for r in layers.values():
        del r
    vrt = isce3.io.Raster(
        str(topo_vrt),
        [isce3.io.Raster(str(out_dir / f"{n}.tif")) for n in ("x", "y", "z")],
    )
    vrt.set_epsg(EPSG)
    del vrt
    return topo_vrt


def run_geo2rdr(sec_burst, topo_vrt: Path, out_dir: Path) -> tuple[Path, Path]:
    out_dir.mkdir(parents=True, exist_ok=True)
    rg_off = out_dir / "range.off"
    az_off = out_dir / "azimuth.off"
    if rg_off.exists() and az_off.exists():
        return rg_off, az_off
    geo2rdr = isce3.geometry.Geo2Rdr(
        sec_burst.as_isce3_radargrid(),
        sec_burst.orbit,
        ELLIPSOID,
        isce3.core.LUT2d(),
        threshold=1e-8,
        numiter=25,
        lines_per_block=1000,
    )
    geo2rdr.geo2rdr(isce3.io.Raster(str(topo_vrt)), str(out_dir))
    return rg_off, az_off


def run_resample(
    sec_burst, ref_grid, rg_off: Path, az_off: Path, out_dir: Path
) -> Path:
    out_dir.mkdir(parents=True, exist_ok=True)
    sec_slc_vrt = out_dir / "secondary.slc.vrt"
    coreg_slc = out_dir / "secondary.coreg.slc.tif"
    if coreg_slc.exists():
        return coreg_slc
    sec_burst.slc_to_vrt_file(str(sec_slc_vrt))
    resamp = isce3.image.ResampSlc(
        sec_burst.as_isce3_radargrid(),
        sec_burst.doppler.lut2d,
        sec_burst.get_az_carrier_poly(),
        ref_rdr_grid=ref_grid,
    )
    resamp.lines_per_tile = 1000
    rg_off_raster = isce3.io.Raster(str(rg_off))
    out_raster = isce3.io.Raster(
        str(coreg_slc),
        rg_off_raster.width,
        rg_off_raster.length,
        1,
        gdal.GDT_CFloat32,
        "GTiff",
    )
    resamp.resamp(
        isce3.io.Raster(str(sec_slc_vrt)),
        out_raster,
        rg_off_raster,
        isce3.io.Raster(str(az_off)),
        flatten=True,
    )
    del out_raster
    return coreg_slc

Loop over the 4 burst pairs and form a per-burst wrapped interferogram. Each burst lands in its own subdirectory under `processing/<burst_id>/` so re-running the notebook only re-does work that's missing.

In [ ]:
import numpy as np
import rasterio

burst_ifgs = []  # list of (burst, ifg_array) in azimuth-time order

for ref_burst, sec_burst in burst_pairs:
    bid = str(ref_burst.burst_id)
    bdir = PROC_DIR / bid
    print(f"\n--- burst {bid} ---")

    topo_vrt = run_rdr2geo(ref_burst, bdir)
    rg_off, az_off = run_geo2rdr(sec_burst, topo_vrt, bdir)
    coreg_slc = run_resample(
        sec_burst, ref_burst.as_isce3_radargrid(), rg_off, az_off, bdir
    )

    with rasterio.open(bdir / "reference.slc.tif") as src:
        ref_arr = src.read(1)
    with rasterio.open(coreg_slc) as src:
        sec_arr = src.read(1)
    # Match shapes: resamp output may differ by a row/col due to offset rounding.
    rows = min(ref_arr.shape[0], sec_arr.shape[0])
    cols = min(ref_arr.shape[1], sec_arr.shape[1])
    ifg = ref_arr[:rows, :cols] * np.conj(sec_arr[:rows, :cols])
    burst_ifgs.append((ref_burst, ifg))
    print(f"  ifg shape: {ifg.shape}")

## Merge bursts along azimuth

All 4 IW2 bursts share the same range grid (same near-range slant range, same range pixel spacing, same width). They differ in `sensing_start` along azimuth, with overlap of ~50-100 lines between adjacent bursts. To merge:

1. Compute each burst's azimuth-line offset relative to the earliest one as `round((sensing_start - t0) / azimuth_time_interval)`.
2. Allocate a merged array spanning `[0, max(offset + length))`.
3. Drop each burst's interferogram in at its offset; in overlap rows the later-laid burst overwrites the earlier (later-in-time burst wins). For real production work you'd use the per-burst `first_valid_line` / `last_valid_line` from the deramping step instead, but this is good enough for visualization.

Range stays as-is.

In [ ]:
burst_ifgs.sort(key=lambda bi: bi[0].sensing_start)
t0 = burst_ifgs[0][0].sensing_start
az_dt = burst_ifgs[0][0].azimuth_time_interval  # seconds per line

offsets = []
for burst, ifg in burst_ifgs:
    delta_lines = int(round((burst.sensing_start - t0).total_seconds() / az_dt))
    offsets.append(delta_lines)
    print(f"burst {burst.burst_id}: offset={delta_lines}, length={ifg.shape[0]}")

merged_rows = max(off + ifg.shape[0] for off, (_, ifg) in zip(offsets, burst_ifgs))
merged_cols = min(ifg.shape[1] for _, ifg in burst_ifgs)
merged = np.zeros((merged_rows, merged_cols), dtype=np.complex64)
for off, (_, ifg) in zip(offsets, burst_ifgs):
    merged[off : off + ifg.shape[0], :merged_cols] = ifg[:, :merged_cols]
print(f"merged shape: {merged.shape}")

## Multilook and plot

Multilook by 5 in range, 20 in azimuth (Sentinel-1 IW pixel spacing is roughly 2.3 m range x 14.0 m azimuth; this brings us to ~12 m x 280 m, similar in aspect to common IW interferogram products). Plot the wrapped phase with `sweets.plotting.plot_ifg`'s cyclic colormap.

In [ ]:
import matplotlib.pyplot as plt
import sweets.plotting  # noqa: F401  -- registers the "oil_slick" cyclic colormap

RG_LOOKS, AZ_LOOKS = 5, 20

def multilook(arr: np.ndarray, az: int, rg: int) -> np.ndarray:
    rows = (arr.shape[0] // az) * az
    cols = (arr.shape[1] // rg) * rg
    a = arr[:rows, :cols]
    return a.reshape(rows // az, az, cols // rg, rg).mean(axis=(1, 3))

ml = multilook(merged, AZ_LOOKS, RG_LOOKS)
phase = np.angle(ml)
phase[ml == 0] = np.nan

fig, ax = plt.subplots(figsize=(6, 10))
im = ax.imshow(phase, cmap="oil_slick", vmin=-np.pi, vmax=np.pi, aspect="auto")
ax.set_title(
    f"radar-coord ifg, {len(burst_pairs)} IW{SWATH} bursts\n"
    f"{REF_DATE.date()} - {SEC_DATE.date()}, multilook {AZ_LOOKS}x{RG_LOOKS}"
)
ax.set_xlabel("range looks")
ax.set_ylabel("azimuth looks (top = early in pass)")
plt.colorbar(im, ax=ax, label="wrapped phase [rad]", shrink=0.7)
out_png = WORK_DIR / "radar_coord_ifg.png"
fig.savefig(out_png, dpi=150, bbox_inches="tight")
print(f"wrote {out_png}")

## Where to go from here

- **ESD refinement**: real burst-merging in topsApp.py uses enhanced spectral diversity over the burst-overlap region to refine the constant azimuth-offset term. ISCE3 has the primitives (`isce3.matchtemplate`, ampcor) to do the same; you'd add a step between `geo2rdr` and `resamp_slc`.
- **Topo-phase removal beyond `flatten`**: `ResampSlc.resamp(..., flatten=True)` removes the geometric range-delay phase ramp but not the residual topographic phase. For deformation-grade interferograms you'd subtract a simulated topo phase computed from the DEM and the perpendicular baseline.
- **More bursts / more dates**: extend the AOI in latitude or relax the `swaths` filter to add IW1/IW3 and merge across subswaths. Each new subswath gets its own range grid, so cross-subswath merging needs a range-direction stitch (or a final geocode).
- **Promote to a workflow**: if this becomes a recurring need it could move into `sweets/_radar_workflow.py` alongside the existing geocoded path. The choice would surface as a new `coordinate_system: radar | geo` knob on the top-level config.